# Phase 6.5 shard 03 (core65)

Runs **30 cells** of the frozen Phase 6.5 manifest (`G3-PHASE65-v1`), covering: `cwdb_dr`, `cwdb_r3_cvridge`, `cwdb_zipt`.

This shard runs the pure-Python methods: the cross-fitted R3 booster, the doubly-robust calibration layer, and the two-part assembly. There is no install step, so computing starts immediately.

Estimated single-threaded compute on the reference machine is about **84 minutes**. Colab cores are slower, so allow two to three times that, plus any install time above. This fits comfortably inside a nine hour session.

**Run every cell in order.** The last cell downloads a `.zip`; collect every shard's zip into `results/phase65/colab_shards/` (logs into `results/manifests/`) and run `python research/run_phase65.py merge`.


In [ ]:
# Thread pinning MUST happen before NumPy or SciPy are imported.
# OpenMP sizes its pool at initialisation, so setting these
# afterwards is silently ineffective.
import os
for _v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS',
           'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS',
           'VECLIB_MAXIMUM_THREADS', 'R_NUM_THREADS'):
    os.environ[_v] = '1'
print('threads pinned to 1')

## 1. Unpack the source tree

In [ ]:
import base64, io, tarfile, pathlib

ARCHIVE = '''...'''

workdir = pathlib.Path('/content/wcf')
workdir.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(ARCHIVE))) as tar:
    tar.extractall(workdir)
os.chdir(workdir)
import sys
sys.path.insert(0, str(workdir / 'src'))
print('source ready at', workdir)

## 2. Dependencies

In [ ]:
# This group needs only NumPy, SciPy, scikit-learn and PyArrow, all
# preinstalled on Colab. Nothing to install.
!python -c "import numpy, scipy, sklearn, pyarrow; print('numpy', numpy.__version__, '| sklearn', sklearn.__version__)"


## 3. This shard's cells

In [ ]:
import json, collections
SHARD_INDEX = 3
CELLS = json.loads('''[{"grid": "c_scaling", "dgp": "IC0", "n_train": 4000, "n_grid": 25, "n_particles": 10, "method": "cwdb_r3_cvridge", "seed": 3, "cell_key": "58372f809392a3bd", "test_seed": 900003}, {"grid": "c_scaling", "dgp": "IC1", "n_train": 2000, "n_grid": 25, "n_particles": 10, "method": "cwdb_r3_cvridge", "seed": 2, "cell_key": "01aa9a158c3e2381", "test_seed": 900002}, {"grid": "d_ablation", "dgp": "DArand", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr", "seed": 5, "cell_key": "7b6583863a76e38f", "test_seed": 900005}, {"grid": "d_ablation", "dgp": "DAunit", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr", "seed": 7, "cell_key": "2829972d72a3d4c4", "test_seed": 900007}, {"grid": "d_ablation", "dgp": "DAref", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr", "seed": 9, "cell_key": "5e48861e0122987b", "test_seed": 900009}, {"grid": "e_zi", "dgp": "ZI0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr", "seed": 1, "cell_key": "a0c153fe3acc811a", "test_seed": 900001}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr", "seed": 3, "cell_key": "49182eb454fc253f", "test_seed": 900003}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr", "seed": 5, "cell_key": "4d4cd040e5fc4d18", "test_seed": 900005}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr", "seed": 7, "cell_key": "93e3b33d59d16e31", "test_seed": 900007}, {"grid": "e_zi", "dgp": "ZI0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr", "seed": 9, "cell_key": "9011018119573bf4", "test_seed": 900009}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr", "seed": 1, "cell_key": "d6bb94fd3637deb1", "test_seed": 900001}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr", "seed": 3, "cell_key": "eb11a6691748aea2", "test_seed": 900003}, {"grid": "d_ablation", "dgp": "DAskew", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_r3_cvridge", "seed": 5, "cell_key": "5a22dc4953844804", "test_seed": 900005}, {"grid": "d_ablation", "dgp": "DArand", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_r3_cvridge", "seed": 7, "cell_key": "2022944be48a4fef", "test_seed": 900007}, {"grid": "d_ablation", "dgp": "DAunit", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_r3_cvridge", "seed": 9, "cell_key": "ca0c71ad42441d0e", "test_seed": 900009}, {"grid": "d_ablation", "dgp": "DAdim", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_r3_cvridge", "seed": 1, "cell_key": "0d4ad9329adcd532", "test_seed": 900001}, {"grid": "e_zi", "dgp": "ZI0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_r3_cvridge", "seed": 3, "cell_key": "d6aa8f06d1a6d23f", "test_seed": 900003}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_r3_cvridge", "seed": 5, "cell_key": "65ec2396175d63c0", "test_seed": 900005}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_r3_cvridge", "seed": 7, "cell_key": "82ee961eba34ffad", "test_seed": 900007}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_r3_cvridge", "seed": 9, "cell_key": "05c1214157744ef7", "test_seed": 900009}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_zipt", "seed": 3, "cell_key": "2655394cb303cfba", "test_seed": 900003}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_zipt", "seed": 5, "cell_key": "ac8e281dfea4a028", "test_seed": 900005}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_zipt", "seed": 7, "cell_key": "b70d0997052fa161", "test_seed": 900007}, {"grid": "e_zi", "dgp": "ZI0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "cwdb_zipt", "seed": 9, "cell_key": "7f2e23a65a264cb0", "test_seed": 900009}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "cwdb_zipt", "seed": 1, "cell_key": "dedb2b496d3d1154", "test_seed": 900001}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "cwdb_zipt", "seed": 3, "cell_key": "8d20f7b800e54c21", "test_seed": 900003}, {"grid": "e_zi", "dgp": "ZI0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "cwdb_r3_cvridge", "seed": 5, "cell_key": "fca4c928a3214dd4", "test_seed": 900005}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "cwdb_r3_cvridge", "seed": 3, "cell_key": "219258bc730cef6f", "test_seed": 900003}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "cwdb_r3_cvridge", "seed": 5, "cell_key": "78d5a2e745a3923c", "test_seed": 900005}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "cwdb_r3_cvridge", "seed": 7, "cell_key": "dbc70f52978e60a7", "test_seed": 900007}]''')
print(f'{len(CELLS)} cells in this shard')
for key, count in sorted(collections.Counter(
        (c['grid'], c['dgp'], c['method'])
        for c in CELLS).items()):
    print(f'  {key[0]:12s} {key[1]:8s} {key[2]:18s} {count}')

## 4. Run

In [ ]:
import time
from pathlib import Path
from wasserstein_causal_forests.g3.manifest import Cell
from wasserstein_causal_forests.g3.runner import run_shard

cells = [Cell(**{k: v for k, v in item.items()
                 if k not in ('cell_key', 'test_seed')})
         for item in CELLS]

out = Path('/content/wcf/results/phase65/colab_shards')
out.mkdir(parents=True, exist_ok=True)
log = Path(f'/content/wcf/results/manifests/phase65_execution_log_{SHARD_INDEX:03d}.jsonl')
log.parent.mkdir(parents=True, exist_ok=True)
cache = Path('/content/wcf/results/rcpp_cache')
cache.mkdir(parents=True, exist_ok=True)

started = time.time()
summary = run_shard(
    cells,
    out / f'shard_{SHARD_INDEX:03d}.parquet',
    cache_directory=cache,
    log_path=log,
    manifest_contract_id='G3-PHASE65-v1',
)
print(json.dumps(summary, indent=2))
print(f'elapsed {(time.time() - started) / 60:.1f} min')

## Check

Every cell must appear exactly once, as a success or as a failure. Failures are kept and reported at merge time; a seed is never silently replaced.

In [ ]:
import collections
records = [json.loads(line) for line in
           open(log, encoding='utf-8') if line.strip()]
status = collections.Counter(r['status'] for r in records)
print('cells logged:', len(records), '| expected:', len(CELLS))
print('status:', dict(status))
assert len(records) == len(CELLS), 'shard did not finish every cell'
for record in records:
    if record['status'] != 'ok':
        print('  FAILED', record['dgp'], record['method'],
              record['seed'])
slowest = sorted(records, key=lambda r: -r['wall_seconds'])[:5]
print('slowest cells:', [(r['method'], round(r['wall_seconds'], 1))
                         for r in slowest])

## Download the results

In [ ]:
import shutil
bundle = '/content/p65_shard_03_core65'
staging = Path('/content/bundle')
if staging.exists():
    shutil.rmtree(staging)
staging.mkdir(parents=True)
shutil.copy(out / f'shard_{SHARD_INDEX:03d}.parquet', staging)
if log.exists():
    shutil.copy(log, staging)
output_file = shutil.make_archive(bundle, 'zip', staging)
print('bundle:', output_file,
      f'({os.path.getsize(output_file) / 1e6:.2f} MB)')

try:
    from google.colab import files
    files.download(output_file)
    print('Downloaded:', output_file)
except Exception as e:
    print('(Not on Colab / download skipped):', e)